# Phase 3 — Patent FAISS avg 추가 (18d + 1024d DualProjection)

**Phase 2와 동일 조건 + Patent 노드에 BGE-M3 임베딩(1024d) 추가**

| | 값 |
|---|---|
| Patent 피처 | 18d (구조) + 1024d (FAISS avg) = **1042d** |
| 차원 불균형 보정 | **DualProjection**: 18d→H//2, 1024d→H//2 → concat → H |
| Company 피처 | FAISS avg 1024d (Phase 2 캐시 재활용) |
| 나머지 조건 | Phase 2 완전 동일 (loss, sampler, split, seed=42) |

### 실험 구성
| 실험 | 방법 | 핵심 |
|---|---|---|
| **실험 1** | **DualProjection** | 18d→64d, 1024d→64d → concat 128d — 모달리티별 독립 경로 (권장) |
| **실험 2** | **Simple Concat** | Linear(1042, 128) — 차원 불균형 보정 없음 (ablation) |

### DualProjection 설계 근거
```
문제: Linear(1042, H) 사용 시
  - 1024d semantic 쪽 파라미터 수 = 18d의 57배
  - Adam optimizer의 gradient 통계가 semantic 쪽에 쏠림
  - 18d 구조 피처의 학습 신호가 묻힘

해결: DualProjection
  18d  struct → Linear(18,   H//2) → LeakyReLU → 64d ─┐
                                                        cat(128d) → GNN
  1024d sem   → Linear(1024, H//2) → LeakyReLU → 64d ─┘
  → 두 모달리티에 동등한 hidden budget 보장
```

## 0. 환경 확인

In [1]:
# cell 1
import dgl
import torch
import torch.nn as nn
import torch.nn.functional as F
import dgl.nn.pytorch as dglnn
import dgl.function as fn
import numpy as np
import pandas as pd
import pickle
import faiss
import time
import warnings
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support,
)

warnings.filterwarnings('ignore', category=UserWarning)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'DGL:     {dgl.__version__}')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if device.type == 'cuda':
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

DGL:     2.4.0+cu124
PyTorch: 2.4.0+cu124
Device:  cuda
GPU:  NVIDIA RTX A6000
VRAM: 51.0 GB


## 1. 경로 설정

In [2]:
# cell 2
BASE_DIR  = Path('/home2/jmbyeon/developments/scratch/patent_eval/uspto_db')
SRC_DIR   = BASE_DIR / 'src'
CSV_DIR   = SRC_DIR  / 'csv_2016'
DATA_DIR  = BASE_DIR / 'data'
EMBED_DIR = DATA_DIR / 'embeddings'

# Phase 1 캐시 재활용
P1_CACHE     = BASE_DIR / '0626' / 'exp' / 'cache_fullgraph'
META_CACHE   = P1_CACHE / 'meta_fullgraph.pkl'
FEAT18_CACHE = P1_CACHE / 'patent_feat_18d.npy'
CITE_CACHE   = P1_CACHE / 'cites_edge.pt'

# Phase 2 캐시 재활용
P2_CACHE      = BASE_DIR / '0626' / 'exp' / 'cache_hetero'
CO_FEAT_CACHE = P2_CACHE / 'company_feat.npy'
CO_META_CACHE = P2_CACHE / 'company_meta.pkl'
OWNS_CACHE    = P2_CACHE / 'owns_edge.pt'
SUES_CACHE    = P2_CACHE / 'sues_edge.pt'

# Phase 3 신규 캐시
P3_CACHE        = BASE_DIR / '0701' / 'exp' / 'cache_phase3'
P3_CACHE.mkdir(parents=True, exist_ok=True)
FAISS_PAT_CACHE = P3_CACHE / 'patent_faiss_avg_f16.npy'  # float16, ~12.5 GB

# 원본
SIM_CSV    = DATA_DIR / 'att_def_top1_sim_v2.csv'
INDEX_PATH = EMBED_DIR / 'faiss.index'
META_PATH  = EMBED_DIR / 'patent_index.parquet'

FAISS_DIM = 1024
FEAT_COLS = ['SK','PPS','ET','IE','FIE','ABS',
             'MCD','SCD',
             'TD_A','TD_B','TD_C','TD_D','TD_E','TD_F','TD_G','TD_H',
             'CD','IC']

print('경로 설정 완료')
print(f'  P3 캐시 디렉토리: {P3_CACHE}')

경로 설정 완료
  P3 캐시 디렉토리: /home2/jmbyeon/developments/scratch/patent_eval/uspto_db/0701/exp/cache_phase3


## 2. 특허 메타 + 18d 피처 (Phase 1 캐시 재활용)

In [3]:
# cell 3
print('[CACHE] Phase 1 meta 로드')
with open(META_CACHE, 'rb') as f:
    meta = pickle.load(f)
pid2idx     = meta['pid2idx']
pid2idx_int = meta['pid2idx_int']
all_patents = meta['all_patents']
n_patents   = meta['n_patents']

print('[CACHE] Phase 1 18d 피처 로드')
patent_feat = np.load(FEAT18_CACHE)  # (6150348, 18) float32

print(f'Patent nodes : {n_patents:,}')
print(f'18d feat     : {patent_feat.shape}  dtype={patent_feat.dtype}')

[CACHE] Phase 1 meta 로드
[CACHE] Phase 1 18d 피처 로드
Patent nodes : 6,150,348
18d feat     : (6150348, 18)  dtype=float32


## 3. Company 노드/피처/엣지 (Phase 2 캐시 재활용)

In [4]:
# cell 4
print('[CACHE] Phase 2 company meta + feat 로드')
with open(CO_META_CACHE, 'rb') as f:
    co_meta = pickle.load(f)
co2idx      = co_meta['co2idx']
all_cos     = co_meta['all_cos']
all_owns    = co_meta['all_owns']
n_companies = len(all_cos)
co_feat     = np.load(CO_FEAT_CACHE)  # (359479, 1024) float32

print(f'Company nodes : {n_companies:,}')
print(f'Company feat  : {co_feat.shape}  dtype={co_feat.dtype}')
print(f'Owns 매핑      : {len(all_owns):,}개 특허')

[CACHE] Phase 2 company meta + feat 로드
Company nodes : 359,479
Company feat  : (359479, 1024)  dtype=float32
Owns 매핑      : 5,324,667개 특허


In [5]:
# cell 5
print('[CACHE] Phase 2 company 엣지 로드')
owns_edge = torch.load(OWNS_CACHE, weights_only=False)
sues_edge = torch.load(SUES_CACHE, weights_only=False)
print(f'owns (co→pat) : {owns_edge.shape[1]:,}개')
print(f'sues (co→co)  : {sues_edge.shape[1]:,}개')

[CACHE] Phase 2 company 엣지 로드
owns (co→pat) : 5,324,632개
sues (co→co)  : 37,712개


## 4. Patent FAISS avg 1024d — 신규

- `patent_index.parquet`: `faiss_id`, `patent_id` 컬럼
- 대부분의 특허는 FAISS 벡터 1개 (78K개만 2~3개 → avg)
- **float16으로 저장** → 캐시 ~12.5 GB (vs float32 25 GB)
- DGL 그래프에는 `'faiss'` 키로 별도 저장, forward 시 `.float()` 변환

In [6]:
# cell 6
if FAISS_PAT_CACHE.exists():
    print('[CACHE] patent FAISS avg 로드 (float16)')
    patent_faiss = np.load(FAISS_PAT_CACHE)
else:
    print('FAISS 인덱스 로드 중... (25 GB, 수 분 소요)')
    index = faiss.read_index(str(INDEX_PATH))
    index.make_direct_map()
    print('  → 인덱스 로드 완료')

    meta_df = pd.read_parquet(META_PATH)
    meta_df['patent_id'] = meta_df['patent_id'].astype(str).str.strip()

    # patent_id → faiss_id 목록
    pid_to_fidxs = defaultdict(list)
    for row in meta_df.itertuples(index=False):
        if row.patent_id in pid2idx:
            pid_to_fidxs[row.patent_id].append(row.faiss_id)
    print(f'  FAISS 커버 특허: {len(pid_to_fidxs):,} / {n_patents:,}')
    del meta_df

    patent_faiss   = np.zeros((n_patents, FAISS_DIM), dtype=np.float16)
    all_pids_sorted = sorted(pid_to_fidxs.keys(), key=lambda p: pid2idx[p])
    BATCH   = 10_000
    n_batch = (len(all_pids_sorted) + BATCH - 1) // BATCH
    t0      = time.time()

    for b in range(n_batch):
        batch_pids = all_pids_sorted[b * BATCH : (b + 1) * BATCH]
        all_fidxs, spans = [], []
        for pid in batch_pids:
            s = len(all_fidxs)
            all_fidxs.extend(pid_to_fidxs[pid])
            spans.append((s, len(all_fidxs)))

        vecs = index.reconstruct_batch(np.array(all_fidxs, dtype=np.int64))
        for j, pid in enumerate(batch_pids):
            s, e = spans[j]
            patent_faiss[pid2idx[pid]] = vecs[s:e].mean(axis=0).astype(np.float16)

        if (b + 1) % 100 == 0 or b == n_batch - 1:
            elapsed = time.time() - t0
            print(f'  [{b+1:4d}/{n_batch}] {(b+1)*BATCH:,} / {len(all_pids_sorted):,}  '
                  f'({elapsed/60:.1f}min)')

    np.save(FAISS_PAT_CACHE, patent_faiss)
    sz = FAISS_PAT_CACHE.stat().st_size / 1e9
    print(f'[SAVED] patent_faiss_avg_f16.npy  {patent_faiss.shape}  {sz:.1f} GB')
    del index

coverage = (patent_faiss.any(axis=1)).mean() * 100
print(f'patent_faiss : {patent_faiss.shape}  dtype={patent_faiss.dtype}')
print(f'FAISS 커버    : {coverage:.1f}%')

[CACHE] patent FAISS avg 로드 (float16)
patent_faiss : (6150348, 1024)  dtype=float16
FAISS 커버    : 100.0%


## 5. Sued 엣지 + Train/Val/Test 분할 (70/10/20) — Phase 1/2 완전 동일

In [7]:
# cell 7
sim_df = pd.read_csv(SIM_CSV, dtype=str)

def make_sued_tensor(df):
    src  = df['att_patent_id'].map(pid2idx)
    dst  = df['def_patent_id'].map(pid2idx)
    mask = src.notna() & dst.notna()
    return torch.tensor(
        np.stack([src[mask].astype(int).values,
                  dst[mask].astype(int).values]),
        dtype=torch.long)

all_pairs = sim_df[['att_patent_id', 'def_patent_id']].drop_duplicates()
all_pairs = all_pairs[
    all_pairs['att_patent_id'].isin(pid2idx) &
    all_pairs['def_patent_id'].isin(pid2idx)
].reset_index(drop=True)

rng     = np.random.default_rng(42)
idx     = rng.permutation(len(all_pairs))
n_train = int(len(idx) * 0.7)
n_val   = int(len(idx) * 0.1)

sued_train = make_sued_tensor(all_pairs.iloc[idx[:n_train]])
sued_val   = make_sued_tensor(all_pairs.iloc[idx[n_train:n_train + n_val]])
sued_test  = make_sued_tensor(all_pairs.iloc[idx[n_train + n_val:]])

print(f'전체 unique 쌍 : {len(all_pairs):,}  (Phase 1/2 동일)')
print(f'train : {sued_train.shape[1]:,}  val : {sued_val.shape[1]:,}  test : {sued_test.shape[1]:,}')

전체 unique 쌍 : 89,938  (Phase 1/2 동일)
train : 62,956  val : 8,993  test : 17,989


## 6. Citation 엣지 — Phase 1 캐시 재활용

In [ ]:
# cell 8
print('[CACHE] Phase 1 citation 엣지 로드')
cites_edge = torch.load(CITE_CACHE, weights_only=False)
print(f'Citation edges : {cites_edge.shape[1]:,}')

[CACHE] Phase 1 citation 엣지 로드
Citation edges : 65,557,586


## 7. 데이터 통계

In [ ]:
# cell 9
n_sued = sued_train.shape[1] + sued_val.shape[1] + sued_test.shape[1]
print('=' * 62)
print(f'  Patent nodes     : {n_patents:>12,}')
print(f'  Company nodes    : {n_companies:>12,}')
print()
print(f'  Patent feat      : {len(FEAT_COLS)}d struct + {FAISS_DIM}d semantic = {len(FEAT_COLS)+FAISS_DIM}d total')
print(f'  Company feat     : {co_feat.shape[1]}d (FAISS avg, Phase 2 동일)')
print()
print(f'  Citation edges   : {cites_edge.shape[1]:>12,}  (patent↔patent)')
print(f'  Sued edges       : {n_sued:>12,}  (patent→patent)')
print(f'  Owns edges       : {owns_edge.shape[1]:>12,}  (company→patent)')
print(f'  Sues edges       : {sues_edge.shape[1]:>12,}  (company→company)')
print('=' * 62)
print(f'  train: {sued_train.shape[1]:,}  val: {sued_val.shape[1]:,}  test: {sued_test.shape[1]:,}')

  Patent nodes     :    6,150,348
  Company nodes    :      359,479

  Patent feat      : 18d struct + 1024d semantic = 1042d total
  Company feat     : 1024d (FAISS avg, Phase 2 동일)

  Citation edges   :   65,557,586  (patent↔patent)
  Sued edges       :       89,938  (patent→patent)
  Owns edges       :    5,324,632  (company→patent)
  Sues edges       :       37,712  (company→company)
  train: 62,956  val: 8,993  test: 17,989


## 8. DGL 헤테로그래프 구성

Patent 노드에 피처 두 개 분리 저장:
- `'feat'` : 18d float32 (구조 피처, 0.44 GB)
- `'faiss'`: 1024d **float16** (의미 피처, ~12.5 GB — float32 대비 절반)

→ forward 시 `build_x_dict()` 에서 concat + float32 변환

In [ ]:
# cell 10
graph_data = {
    ('patent',  'cites',    'patent'):  (cites_edge[0], cites_edge[1]),
    ('patent',  'cited_by', 'patent'):  (cites_edge[1], cites_edge[0]),
    ('patent',  'sued',     'patent'):  (sued_train[0], sued_train[1]),
    ('patent',  'sued_by',  'patent'):  (sued_train[1], sued_train[0]),
    ('company', 'owns',     'patent'):  (owns_edge[0],  owns_edge[1]),
    ('patent',  'owned_by', 'company'): (owns_edge[1],  owns_edge[0]),
    ('company', 'sues',     'company'): (sues_edge[0],  sues_edge[1]),
    ('company', 'sued_by',  'company'): (sues_edge[1],  sues_edge[0]),
}

g = dgl.heterograph(graph_data, num_nodes_dict={
    'patent':  n_patents,
    'company': n_companies,
})

g.nodes['patent'].data['feat']  = torch.tensor(patent_feat,  dtype=torch.float32)
g.nodes['patent'].data['faiss'] = torch.from_numpy(patent_faiss)            # float16
g.nodes['company'].data['feat'] = torch.tensor(co_feat,      dtype=torch.float32)

print(g)
print()
print(f"Patent  'feat'  : {g.nodes['patent'].data['feat'].shape}   {g.nodes['patent'].data['feat'].dtype}")
print(f"Patent  'faiss' : {g.nodes['patent'].data['faiss'].shape}  {g.nodes['patent'].data['faiss'].dtype}")
print(f"Company 'feat'  : {g.nodes['company'].data['feat'].shape}  {g.nodes['company'].data['feat'].dtype}")

Graph(num_nodes={'company': 359479, 'patent': 6150348},
      num_edges={('company', 'owns', 'patent'): 5324632, ('company', 'sued_by', 'company'): 37712, ('company', 'sues', 'company'): 37712, ('patent', 'cited_by', 'patent'): 65557586, ('patent', 'cites', 'patent'): 65557586, ('patent', 'owned_by', 'company'): 5324632, ('patent', 'sued', 'patent'): 62956, ('patent', 'sued_by', 'patent'): 62956},
      metagraph=[('company', 'patent', 'owns'), ('company', 'company', 'sued_by'), ('company', 'company', 'sues'), ('patent', 'patent', 'cited_by'), ('patent', 'patent', 'cites'), ('patent', 'patent', 'sued'), ('patent', 'patent', 'sued_by'), ('patent', 'company', 'owned_by')])

Patent  'feat'  : torch.Size([6150348, 18])   torch.float32
Patent  'faiss' : torch.Size([6150348, 1024])  torch.float16
Company 'feat'  : torch.Size([359479, 1024])  torch.float32


## 9. 공통 유틸 (build_x_dict / get_embeddings / eval_metrics)

In [ ]:
# cell 11
N_LAYERS = 4
K        = 2


def build_x_dict(block):
    # patent: 18d float32 + 1024d float16 → cat → 1042d float32
    # company: 1024d float32
    x_dict = {}
    for ntype in block.srctypes:
        if block.num_src_nodes(ntype) == 0:
            continue
        if ntype == 'patent':
            struct = block.srcnodes['patent'].data['feat']           # 18d float32
            sem    = block.srcnodes['patent'].data['faiss'].float()  # 1024d → float32
            x_dict['patent'] = torch.cat([struct, sem], dim=1)      # 1042d
        else:
            x_dict[ntype] = block.srcnodes[ntype].data['feat']
    return x_dict


def get_embeddings(model, g, nids, device):
    model.eval()
    sampler = dgl.dataloading.NeighborSampler([K] * N_LAYERS)
    loader  = dgl.dataloading.DataLoader(
        g, {'patent': torch.tensor(nids, dtype=torch.long)},
        sampler, batch_size=512, shuffle=False, num_workers=0, device='cpu'
    )
    buf = {}
    with torch.no_grad():
        for _, out_nodes, blocks in loader:
            blocks = [b.to(device) for b in blocks]
            x_dict = build_x_dict(blocks[0])
            h      = model.rgcn(blocks, x_dict)
            for local_i, gid in enumerate(out_nodes['patent'].tolist()):
                buf[gid] = h['patent'][local_i].detach().cpu().numpy()
    return buf


def eval_metrics(model, g, src_ids, dst_ids, device, best_thresh=None):
    from numpy.linalg import norm
    all_ids  = np.unique(np.concatenate([src_ids, dst_ids]))
    emb_dict = get_embeddings(model, g, all_ids, device)

    src_e   = np.stack([emb_dict[i] for i in src_ids])
    dst_e   = np.stack([emb_dict[i] for i in dst_ids])
    pos_sim = (src_e * dst_e).sum(1) / (norm(src_e, axis=1) * norm(dst_e, axis=1) + 1e-8)

    rng_eval = np.random.default_rng(99)
    neg_ids  = rng_eval.integers(0, n_patents, size=len(src_ids))
    neg_u    = np.unique(neg_ids)
    neg_dict = get_embeddings(model, g, neg_u, device)
    neg_e    = np.stack([neg_dict[i] for i in neg_ids])
    neg_sim  = (src_e * neg_e).sum(1) / (norm(src_e, axis=1) * norm(neg_e, axis=1) + 1e-8)

    sims   = np.concatenate([pos_sim, neg_sim])
    labels = np.array([1] * len(pos_sim) + [0] * len(neg_sim))

    auroc = roc_auc_score(labels, sims)
    auprc = average_precision_score(labels, sims)

    median      = np.median(sims)
    best_f1_cur = 0
    best_t_cur  = median
    for i in range(-50, 50):
        t    = (best_thresh + 0.001 * i) if best_thresh is not None else (median + 0.001 * i)
        pred = (sims >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            labels, pred, average='binary', zero_division=0)
        if f1 > best_f1_cur:
            best_f1_cur, best_t_cur = f1, t

    return auroc, auprc, best_f1_cur, best_t_cur


def margin_loss(pos_score, neg_score, etype):
    n = pos_score[etype].shape[0]
    return (1 - pos_score[etype].unsqueeze(1)
              + neg_score[etype].view(n, -1)).clamp(min=0).mean()


TARGET_ETYPE = ('patent', 'sued', 'patent')
PAT_DIM      = patent_feat.shape[1]       # 18
CO_DIM       = co_feat.shape[1]           # 1024
PAT_TOTAL    = PAT_DIM + FAISS_DIM        # 1042
HID_DIM      = 128
OUT_DIM      = 64
rel_names    = [e[1] for e in g.canonical_etypes]

print('공통 유틸 정의 완료')
print(f'  rel_names  : {rel_names}')
print(f'  PAT_TOTAL  : {PAT_TOTAL}d  ({PAT_DIM}d struct + {FAISS_DIM}d sem)')
print(f'  HID/OUT    : {HID_DIM} / {OUT_DIM}')

공통 유틸 정의 완료
  rel_names  : ['owns', 'sued_by', 'sues', 'cited_by', 'cites', 'owned_by', 'sued', 'sued_by']
  PAT_TOTAL  : 1042d  (18d struct + 1024d sem)
  HID/OUT    : 128 / 64


## 공통 모델 구조

`input_proj_patent` 인자만 교체하면 실험 1/2 모두 동일한 `HeteroRGCN` 사용.

In [ ]:
# cell 12
class HeteroRGCN(nn.Module):
    def __init__(self, co_in, hid, out, rel_names, input_proj_patent):
        super().__init__()
        self.hid = hid
        self.out = out

        # patent: DualProjection 또는 SimpleLinearProj (실험별로 교체)
        # company: 항상 Linear(co_in, hid)
        self.input_proj = nn.ModuleDict({
            'patent':  input_proj_patent,
            'company': nn.Linear(co_in, hid),
        })

        self.conv1 = dglnn.HeteroGraphConv(
            {r: dglnn.GATConv(hid, hid // 8, 8, attn_drop=0.3) for r in rel_names},
            aggregate='mean')
        self.conv2 = dglnn.HeteroGraphConv(
            {r: dglnn.GATConv(hid, hid // 4, 4) for r in rel_names},
            aggregate='mean')
        self.conv3 = dglnn.HeteroGraphConv(
            {r: dglnn.GATConv(hid, hid // 4, 4) for r in rel_names},
            aggregate='mean')
        self.conv4 = dglnn.HeteroGraphConv(
            {r: dglnn.GATConv(hid, out // 2, 2) for r in rel_names},
            aggregate='mean')

        self.bns = nn.ModuleDict({
            'patent':  nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(3)]),
            'company': nn.ModuleList([nn.BatchNorm1d(hid) for _ in range(3)]),
        })

    def _bn_act(self, h, bn_idx, act):
        result = {}
        for ntype, feat in h.items():
            flat = feat.view(-1, self.hid)
            if ntype in self.bns and flat.shape[0] > 1:
                flat = self.bns[ntype][bn_idx](flat)
            result[ntype] = act(flat)
        return result

    def forward(self, blocks, x_dict):
        h = {ntype: self.input_proj[ntype](feat)
             for ntype, feat in x_dict.items()
             if ntype in self.input_proj}
        h = self.conv1(blocks[0], h)
        h = self._bn_act(h, 0, F.leaky_relu)
        h = self.conv2(blocks[1], h)
        h = self._bn_act(h, 1, F.leaky_relu)
        h = self.conv3(blocks[2], h)
        h = self._bn_act(h, 2, torch.tanh)
        h = self.conv4(blocks[3], h)
        return {k: v.view(-1, self.out) for k, v in h.items()}


class ScorePredictor(nn.Module):
    def forward(self, edge_subgraph, h):
        with edge_subgraph.local_scope():
            for ntype, feat in h.items():
                edge_subgraph.nodes[ntype].data['h'] = feat
            for etype in edge_subgraph.canonical_etypes:
                edge_subgraph.apply_edges(fn.u_dot_v('h', 'h', 'score'), etype=etype)
            return edge_subgraph.edata['score']


class PatentLinkModel(nn.Module):
    def __init__(self, co_in, hid, out, rel_names, input_proj_patent):
        super().__init__()
        self.rgcn = HeteroRGCN(co_in, hid, out, rel_names, input_proj_patent)
        self.pred = ScorePredictor()

    def forward(self, pos_g, neg_g, blocks, x_dict):
        h = self.rgcn(blocks, x_dict)
        return self.pred(pos_g, h), self.pred(neg_g, h)


print('공통 모델 클래스 정의 완료 (HeteroRGCN / ScorePredictor / PatentLinkModel)')

공통 모델 클래스 정의 완료 (HeteroRGCN / ScorePredictor / PatentLinkModel)


---
## 실험 1 — DualProjection (18d→64d, 1024d→64d)

```
18d  struct → Linear(18,   64) → LeakyReLU → 64d ─┐
                                                    cat(128d) → GNN
1024d sem   → Linear(1024, 64) → LeakyReLU → 64d ─┘
```

- 구조/의미 피처 각각 **독립적인 64d 경로** 보장
- 18d 구조 피처 gradient가 1024d 쪽에 묻히지 않음
- H=128 유지 (Phase 2 동일)

In [ ]:
# cell 13
class DualProjection(nn.Module):
    # 차원 불균형 보정: 구조(18d)와 의미(1024d)를 동등한 hidden budget(H//2)으로 독립 투영
    def __init__(self, dim_struct, dim_sem, hid, dropout=0.1):
        super().__init__()
        half = hid // 2
        self.proj_struct = nn.Linear(dim_struct, half)
        self.proj_sem    = nn.Linear(dim_sem,    half)
        self.drop        = nn.Dropout(dropout)
        self.dim_struct  = dim_struct

    def forward(self, x):  # x: [N, dim_struct + dim_sem]
        h_s = F.leaky_relu(self.proj_struct(x[:, :self.dim_struct]))
        h_e = F.leaky_relu(self.proj_sem(   x[:,  self.dim_struct:]))
        return self.drop(torch.cat([h_s, h_e], dim=1))  # [N, hid]


dual_proj = DualProjection(dim_struct=PAT_DIM, dim_sem=FAISS_DIM, hid=HID_DIM)

model1 = PatentLinkModel(
    co_in=CO_DIM, hid=HID_DIM, out=OUT_DIM,
    rel_names=rel_names,
    input_proj_patent=dual_proj,
).to(device)

n_params_1 = sum(p.numel() for p in model1.parameters())
print(f'[실험 1] DualProjection')
print(f'  patent 입력 : {PAT_DIM}d struct → Linear({PAT_DIM}, {HID_DIM//2})')
print(f'                {FAISS_DIM}d sem   → Linear({FAISS_DIM}, {HID_DIM//2})')
print(f'                concat → {HID_DIM}d')
print(f'  파라미터 수  : {n_params_1:,}')

[실험 1] DualProjection
  patent 입력 : 18d struct → Linear(18, 64)
                1024d sem   → Linear(1024, 64)
                concat → 128d
  파라미터 수  : 610,368


In [ ]:
# cell 14
BATCH_SIZE = 1024
NEG_K      = 4

train_eid_dict = {TARGET_ETYPE: g.edges(etype='sued', form='eid')}

sampler1 = dgl.dataloading.NeighborSampler([K] * N_LAYERS)
sampler1 = dgl.dataloading.as_edge_prediction_sampler(
    sampler1,
    negative_sampler=dgl.dataloading.negative_sampler.PerSourceUniform(NEG_K),
)
train_loader1 = dgl.dataloading.DataLoader(
    g, train_eid_dict, sampler1,
    device='cpu', batch_size=BATCH_SIZE,
    shuffle=True, drop_last=False, num_workers=0,
)

print(f'DataLoader 준비 (실험 1)')
print(f'  NeighborSampler : {N_LAYERS} layers x {K} neighbors/hop')
print(f'  PerSourceUniform: neg_k={NEG_K}  batch={BATCH_SIZE}')
print(f'  batches/epoch   : ~{sued_train.shape[1]//BATCH_SIZE:,}')

DataLoader 준비 (실험 1)
  NeighborSampler : 4 layers x 4 neighbors/hop
  PerSourceUniform: neg_k=4  batch=1024
  batches/epoch   : ~61


In [ ]:
# cell 15
EPOCHS   = 30
LR       = 1e-3
WD       = 1e-5
PATIENCE = 5

optimizer1 = torch.optim.Adam(model1.parameters(), lr=LR, weight_decay=WD)
scheduler1 = torch.optim.lr_scheduler.StepLR(optimizer1, step_size=10, gamma=0.78)

best_f1_1     = 0
best_thresh_1 = None
best_state_1  = None
no_improve_1  = 0
history_1     = []

print(f'[실험 1] 학습 시작 — DualProjection  EPOCHS={EPOCHS}  LR={LR}  PATIENCE={PATIENCE}')
print('-' * 95)

for epoch in range(1, EPOCHS + 1):
    model1.train()
    losses = []
    t0 = time.time()

    for input_nodes, pos_g, neg_g, blocks in train_loader1:
        blocks = [b.to(device) for b in blocks]
        pos_g  = pos_g.to(device)
        neg_g  = neg_g.to(device)
        x_dict = build_x_dict(blocks[0])

        pos_score, neg_score = model1(pos_g, neg_g, blocks, x_dict)
        loss = margin_loss(pos_score, neg_score, TARGET_ETYPE)
        optimizer1.zero_grad()
        loss.backward()
        optimizer1.step()
        losses.append(loss.item())

    scheduler1.step()

    val_auroc, val_auprc, val_f1, val_thresh = eval_metrics(
        model1, g, sued_val[0].numpy(), sued_val[1].numpy(), device, best_thresh_1)
    history_1.append((epoch, np.mean(losses), val_auroc, val_auprc, val_f1))

    if val_f1 > best_f1_1:
        best_f1_1     = val_f1
        best_thresh_1 = val_thresh
        best_state_1  = {k: v.clone() for k, v in model1.state_dict().items()}
        no_improve_1  = 0
        tag = '★ NEW BEST'
    else:
        no_improve_1 += 1
        tag = f'no improve {no_improve_1}/{PATIENCE}'

    print(f'Epoch {epoch:3d}/{EPOCHS} | loss={np.mean(losses):.4f} '
          f'AUROC={val_auroc:.4f} AUPRC={val_auprc:.4f} F1={val_f1:.4f} '
          f'thresh={val_thresh:.4f}  ({time.time()-t0:.1f}s)  {tag}')

    if no_improve_1 >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

[실험 1] 학습 시작 — DualProjection  EPOCHS=30  LR=0.001  PATIENCE=5
-----------------------------------------------------------------------------------------------


OutOfMemoryError: CUDA out of memory. Tried to allocate 2.69 GiB. GPU 0 has a total capacity of 47.53 GiB of which 821.62 MiB is free. Process 801584 has 6.62 GiB memory in use. Process 3283159 has 2.10 GiB memory in use. Including non-PyTorch memory, this process has 38.00 GiB memory in use. Of the allocated memory 35.89 GiB is allocated by PyTorch, and 1.80 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# cell 16
if best_state_1:
    model1.load_state_dict(best_state_1)

test_auroc_1, test_auprc_1, test_f1_1, _ = eval_metrics(
    model1, g, sued_test[0].numpy(), sued_test[1].numpy(), device, best_thresh_1)

print('=' * 55)
print('[실험 1] DualProjection — Test 결과')
print(f'  AUROC  : {test_auroc_1:.4f}')
print(f'  AUPRC  : {test_auprc_1:.4f}')
print(f'  F1     : {test_f1_1:.4f}')
print(f'  thresh : {best_thresh_1:.4f}')
print('=' * 55)
print(f'  (Phase 2 기준: AUROC=0.9409  F1=0.8592)')

---
## 실험 2 — Simple Concat  Linear(1042, H)  [Ablation]

```
[18d struct | 1024d sem] → Linear(1042, 128) → LeakyReLU → H=128 → GNN
```

- DualProjection 없이 단순 concat → 선형 투영
- 1024d 쪽 파라미터가 18d의 **57배** → 구조 피처 gradient 약화 예상
- 나머지 조건 모두 동일 (같은 DataLoader, optimizer, PATIENCE 설정)
- **목적**: 실험 1 대비 성능 차이로 DualProjection 효과 정량화

In [ ]:
# cell 17
class SimpleLinearProj(nn.Module):
    # 비교용: Linear(1042, H) — 차원 불균형 보정 없음
    def __init__(self, dim_total, hid, dropout=0.1):
        super().__init__()
        self.proj = nn.Linear(dim_total, hid)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return self.drop(F.leaky_relu(self.proj(x)))


simple_proj = SimpleLinearProj(dim_total=PAT_TOTAL, hid=HID_DIM)

model2 = PatentLinkModel(
    co_in=CO_DIM, hid=HID_DIM, out=OUT_DIM,
    rel_names=rel_names,
    input_proj_patent=simple_proj,
).to(device)

n_params_2 = sum(p.numel() for p in model2.parameters())
print(f'[실험 2] Simple Concat')
print(f'  patent 입력 : {PAT_TOTAL}d (18+1024) → Linear({PAT_TOTAL}, {HID_DIM})')
print(f'  파라미터 수  : {n_params_2:,}  (실험 1: {n_params_1:,})')

In [ ]:
# cell 18
# DataLoader: 동일 설정, sampler 새로 생성 (model과 독립)
sampler2 = dgl.dataloading.NeighborSampler([K] * N_LAYERS)
sampler2 = dgl.dataloading.as_edge_prediction_sampler(
    sampler2,
    negative_sampler=dgl.dataloading.negative_sampler.PerSourceUniform(NEG_K),
)
train_loader2 = dgl.dataloading.DataLoader(
    g, train_eid_dict, sampler2,
    device='cpu', batch_size=BATCH_SIZE,
    shuffle=True, drop_last=False, num_workers=0,
)

optimizer2 = torch.optim.Adam(model2.parameters(), lr=LR, weight_decay=WD)
scheduler2 = torch.optim.lr_scheduler.StepLR(optimizer2, step_size=10, gamma=0.78)

best_f1_2     = 0
best_thresh_2 = None
best_state_2  = None
no_improve_2  = 0
history_2     = []

print(f'[실험 2] 학습 시작 — Simple Concat  EPOCHS={EPOCHS}  LR={LR}  PATIENCE={PATIENCE}')
print('-' * 95)

for epoch in range(1, EPOCHS + 1):
    model2.train()
    losses = []
    t0 = time.time()

    for input_nodes, pos_g, neg_g, blocks in train_loader2:
        blocks = [b.to(device) for b in blocks]
        pos_g  = pos_g.to(device)
        neg_g  = neg_g.to(device)
        x_dict = build_x_dict(blocks[0])

        pos_score, neg_score = model2(pos_g, neg_g, blocks, x_dict)
        loss = margin_loss(pos_score, neg_score, TARGET_ETYPE)
        optimizer2.zero_grad()
        loss.backward()
        optimizer2.step()
        losses.append(loss.item())

    scheduler2.step()

    val_auroc, val_auprc, val_f1, val_thresh = eval_metrics(
        model2, g, sued_val[0].numpy(), sued_val[1].numpy(), device, best_thresh_2)
    history_2.append((epoch, np.mean(losses), val_auroc, val_auprc, val_f1))

    if val_f1 > best_f1_2:
        best_f1_2     = val_f1
        best_thresh_2 = val_thresh
        best_state_2  = {k: v.clone() for k, v in model2.state_dict().items()}
        no_improve_2  = 0
        tag = '★ NEW BEST'
    else:
        no_improve_2 += 1
        tag = f'no improve {no_improve_2}/{PATIENCE}'

    print(f'Epoch {epoch:3d}/{EPOCHS} | loss={np.mean(losses):.4f} '
          f'AUROC={val_auroc:.4f} AUPRC={val_auprc:.4f} F1={val_f1:.4f} '
          f'thresh={val_thresh:.4f}  ({time.time()-t0:.1f}s)  {tag}')

    if no_improve_2 >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}')
        break

In [ ]:
# cell 19
if best_state_2:
    model2.load_state_dict(best_state_2)

test_auroc_2, test_auprc_2, test_f1_2, _ = eval_metrics(
    model2, g, sued_test[0].numpy(), sued_test[1].numpy(), device, best_thresh_2)

print('=' * 55)
print('[실험 2] Simple Concat — Test 결과')
print(f'  AUROC  : {test_auroc_2:.4f}')
print(f'  AUPRC  : {test_auprc_2:.4f}')
print(f'  F1     : {test_f1_2:.4f}')
print(f'  thresh : {best_thresh_2:.4f}')
print('=' * 55)

---
## 결과 비교

In [ ]:
# cell 20
rows = [
    ('Phase 1 (homo)',       'homograph',   '18d',   '-',     0.9150, 0.9151, 0.8328),
    ('Phase 2 (hetero)',     'heterograph', '18d',   '1024d', 0.9409, 0.9438, 0.8592),
    ('실험 1: DualProj',    'heterograph', '1042d', '1024d', test_auroc_1, test_auprc_1, test_f1_1),
    ('실험 2: SimpleCat',   'heterograph', '1042d', '1024d', test_auroc_2, test_auprc_2, test_f1_2),
]

hdr = '{:<22} {:<14} {:<8} {:<8} {:>7} {:>7} {:>7}'.format(
    '실험', '구조', 'Patent', 'Company', 'AUROC', 'AUPRC', 'F1')
sep = '=' * 76

print(sep)
print(hdr)
print('-' * 76)
for name, struct, pat, co, auroc, auprc, f1 in rows:
    print('{:<22} {:<14} {:<8} {:<8} {:>7.4f} {:>7.4f} {:>7.4f}'.format(
        name, struct, pat, co, auroc, auprc, f1))
print(sep)

d_dual_vs_p2   = (test_auroc_1 - 0.9409, test_f1_1 - 0.8592)
d_simple_vs_p2 = (test_auroc_2 - 0.9409, test_f1_2 - 0.8592)
d_dual_vs_simple = (test_auroc_1 - test_auroc_2, test_f1_1 - test_f1_2)

print()
print(f'실험 1 (DualProj) vs Phase 2   : AUROC {d_dual_vs_p2[0]:+.4f}  F1 {d_dual_vs_p2[1]:+.4f}')
print(f'실험 2 (SimpleCat) vs Phase 2  : AUROC {d_simple_vs_p2[0]:+.4f}  F1 {d_simple_vs_p2[1]:+.4f}')
print(f'DualProj vs SimpleCat          : AUROC {d_dual_vs_simple[0]:+.4f}  F1 {d_dual_vs_simple[1]:+.4f}')

---
## Feature Importance Analysis

1. **Group Ablation** — struct(18d) vs semantic(1024d) 기여도 분리  
   - `struct_only`: FAISS 임베딩을 0으로 마스킹  
   - `sem_only`: 구조 피처 18d를 0으로 마스킹  
   - SimpleConcatLinear / DualProjection 두 모델 비교

2. **Permutation Importance** — 18개 구조 피처 개별 기여도  
   - 각 피처 컬럼을 전체 노드에 걸쳐 무작위 셔플 후 AUROC 변화 측정  
   - 대상: SimpleConcatLinear (최고 성능 모델)

In [ ]:
# cell fi-1: Ablation 유틸
from numpy.linalg import norm as np_norm

def get_embs_ablation(model, g, nids, device, mode='full', perm_feat_idx=None):
    """
    mode: 'full' | 'struct_only' | 'sem_only' | 'perm'
    perm_feat_idx: 0~17, mode='perm'일 때 셔플할 구조 피처 컬럼 인덱스
    """
    orig_col = None
    if mode == 'perm' and perm_feat_idx is not None:
        orig_col  = g.nodes['patent'].data['feat'][:, perm_feat_idx].clone()
        perm_gen  = torch.Generator()
        perm_gen.manual_seed(0)
        perm_ord  = torch.randperm(g.num_nodes('patent'), generator=perm_gen)
        g.nodes['patent'].data['feat'][:, perm_feat_idx] = orig_col[perm_ord]

    def _build_x(block):
        x = {}
        for ntype in block.srctypes:
            if block.num_src_nodes(ntype) == 0:
                continue
            if ntype == 'patent':
                struct = block.srcnodes['patent'].data['feat']
                sem    = block.srcnodes['patent'].data['faiss'].float()
                if mode == 'struct_only':
                    sem    = torch.zeros_like(sem)
                elif mode == 'sem_only':
                    struct = torch.zeros_like(struct)
                x['patent'] = torch.cat([struct, sem], dim=1)
            else:
                x[ntype] = block.srcnodes[ntype].data['feat']
        return x

    model.eval()
    sampler = dgl.dataloading.NeighborSampler([K] * N_LAYERS)
    loader  = dgl.dataloading.DataLoader(
        g, {'patent': torch.tensor(nids, dtype=torch.long)},
        sampler, batch_size=512, shuffle=False, num_workers=0, device='cpu'
    )
    buf = {}
    with torch.no_grad():
        for _, out_nodes, blocks in loader:
            blocks = [b.to(device) for b in blocks]
            h      = model.rgcn(blocks, _build_x(blocks[0]))
            for local_i, gid in enumerate(out_nodes['patent'].tolist()):
                buf[gid] = h['patent'][local_i].detach().cpu().numpy()

    if orig_col is not None:
        g.nodes['patent'].data['feat'][:, perm_feat_idx] = orig_col

    return buf


def eval_ablation(model, g, src_ids, dst_ids, device, best_thresh,
                  mode='full', perm_feat_idx=None):
    all_ids  = np.unique(np.concatenate([src_ids, dst_ids]))
    emb_dict = get_embs_ablation(model, g, all_ids, device, mode, perm_feat_idx)

    src_e   = np.stack([emb_dict[i] for i in src_ids])
    dst_e   = np.stack([emb_dict[i] for i in dst_ids])
    pos_sim = (src_e * dst_e).sum(1) / (np_norm(src_e, axis=1) * np_norm(dst_e, axis=1) + 1e-8)

    rng_neg = np.random.default_rng(99)
    neg_ids = rng_neg.integers(0, g.num_nodes('patent'), size=len(src_ids))
    neg_u   = np.unique(neg_ids)
    neg_dict = get_embs_ablation(model, g, neg_u, device, mode, perm_feat_idx)
    neg_e   = np.stack([neg_dict[i] for i in neg_ids])
    neg_sim = (src_e * neg_e).sum(1) / (np_norm(src_e, axis=1) * np_norm(neg_e, axis=1) + 1e-8)

    sims   = np.concatenate([pos_sim, neg_sim])
    labels = np.array([1] * len(pos_sim) + [0] * len(neg_sim))

    auroc = roc_auc_score(labels, sims)
    auprc = average_precision_score(labels, sims)

    median = np.median(sims)
    best_f1, best_t = 0, median
    for i in range(-50, 50):
        t    = (best_thresh + 0.001 * i) if best_thresh is not None else (median + 0.001 * i)
        pred = (sims >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            labels, pred, average='binary', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t

    return auroc, auprc, best_f1, best_t


print('eval_ablation / get_embs_ablation 정의 완료')

In [ ]:
# cell fi-2: Group Ablation — struct vs semantic 기여도 (두 모델 비교)
test_src = sued_test[0].numpy()
test_dst = sued_test[1].numpy()

abl_cfg = [
    ('SimpleCat', model2, best_thresh_2),
    ('DualProj',  model1, best_thresh_1),
]
group_results = {}   # (model_name, mode) -> (auroc, auprc, f1)

for model_name, model, thresh in abl_cfg:
    print(f'=== {model_name} ===')
    for mode in ('full', 'struct_only', 'sem_only'):
        t0 = time.time()
        auroc, auprc, f1, _ = eval_ablation(
            model, g, test_src, test_dst, device, thresh, mode=mode)
        group_results[(model_name, mode)] = (auroc, auprc, f1)
        print(f'  [{mode:<12}]  AUROC={auroc:.4f}  AUPRC={auprc:.4f}  F1={f1:.4f}  ({time.time()-t0:.0f}s)')
    print()

# 요약 테이블
print('=' * 70)
print(f'{"모델 / 모드":<24}  {"AUROC":>7}  {"ΔAUROC":>8}  {"F1":>7}  {"ΔF1":>7}')
print('-' * 70)
for model_name, _, _ in abl_cfg:
    base_a, _, base_f = group_results[(model_name, 'full')]
    for mode in ('full', 'struct_only', 'sem_only'):
        auroc, _, f1 = group_results[(model_name, mode)]
        tag = f'{model_name} {mode}'
        print(f'  {tag:<22}  {auroc:>7.4f}  {auroc-base_a:>+8.4f}  {f1:>7.4f}  {f1-base_f:>+7.4f}')
    print()
print('=' * 70)

In [ ]:
# cell fi-3: Permutation Importance — 18 struct features (SimpleConcatLinear)
# 각 피처를 전체 특허 노드에서 셔플 후 AUROC 변화 측정
# ΔAUROC가 클수록(더 음수) 해당 피처가 중요
base_auroc = group_results[('SimpleCat', 'full')][0]
base_f1    = group_results[('SimpleCat', 'full')][2]

print(f'Permutation Importance — SimpleConcatLinear  (baseline AUROC={base_auroc:.4f})')
print(f'test pairs: {len(test_src):,}')
print('-' * 62)

perm_results = []
for i, feat_name in enumerate(FEAT_COLS):
    t0 = time.time()
    auroc, _, f1, _ = eval_ablation(
        model2, g, test_src, test_dst, device, best_thresh_2,
        mode='perm', perm_feat_idx=i
    )
    d_auroc = auroc - base_auroc
    d_f1    = f1    - base_f1
    perm_results.append({'feat': feat_name, 'auroc': auroc, 'd_auroc': d_auroc,
                         'f1': f1, 'd_f1': d_f1})
    print(f'  [{i:2d}] {feat_name:<6}  AUROC={auroc:.4f} ({d_auroc:+.4f})  '
          f'F1={f1:.4f} ({d_f1:+.4f})  ({time.time()-t0:.0f}s)')

# 중요도 순위 (ΔAUROC 낮을수록 = 셔플 시 성능 저하 크다 = 중요)
perm_sorted = sorted(perm_results, key=lambda x: x['d_auroc'])

print()
print('=== Importance Ranking (ΔAUROC ↓ = 중요) ===')
for rank, r in enumerate(perm_sorted, 1):
    bar = '█' * max(0, int(abs(r['d_auroc']) * 5000))
    print(f'  {rank:2d}. {r["feat"]:<6}  {r["d_auroc"]:+.5f}  {bar}')

In [ ]:
# cell fi-4: Visualization
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# ── Panel 1: Group Ablation ──────────────────────────────────────────
ax1 = axes[0]
model_labels = ['SimpleCat', 'DualProj']
modes        = ['full', 'struct_only', 'sem_only']
mode_labels  = ['full', 'struct only\n(sem=0)', 'sem only\n(struct=0)']
colors       = ['#2196F3', '#FF9800', '#4CAF50']
x = np.arange(len(model_labels))
width = 0.24

for j, (mode, lbl, col) in enumerate(zip(modes, mode_labels, colors)):
    vals = [group_results[(m, mode)][0] for m in model_labels]
    bars = ax1.bar(x + (j - 1) * width, vals, width, label=lbl, color=col, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                 f'{v:.3f}', ha='center', va='bottom', fontsize=8.5)

ax1.set_ylim(0.45, 1.02)
ax1.set_xticks(x)
ax1.set_xticklabels(model_labels, fontsize=11)
ax1.set_ylabel('AUROC')
ax1.set_title('Group Ablation: struct vs semantic contribution', fontsize=11)
ax1.legend(fontsize=9)
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax1.yaxis.set_minor_locator(mticker.MultipleLocator(0.05))

# ── Panel 2: Permutation Importance ─────────────────────────────────
ax2 = axes[1]
feat_names   = [r['feat']    for r in perm_sorted]
delta_aurocs = [r['d_auroc'] for r in perm_sorted]
bar_colors   = ['#E53935' if d < -0.0005 else '#90A4AE' for d in delta_aurocs]

bars = ax2.barh(feat_names, delta_aurocs, color=bar_colors, alpha=0.85, edgecolor='white')
ax2.axvline(0, color='black', linewidth=0.8)

for i, (bar, v) in enumerate(zip(bars, delta_aurocs)):
    offset = -0.00008 if v <= 0 else 0.00008
    ha     = 'right'  if v <= 0 else 'left'
    ax2.text(v + offset, bar.get_y() + bar.get_height() / 2,
             f'{v:+.5f}', ha=ha, va='center', fontsize=8)

ax2.set_xlabel('ΔAUROC  (permuted − baseline,  더 음수 = 더 중요)')
ax2.set_title('Permutation Importance — 18 struct features\n(SimpleConcatLinear)', fontsize=11)
ax2.invert_yaxis()

plt.tight_layout()
out_path = 'feature_importance.png'
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')